# RiffSpace: Quick Start Guide

This notebook demonstrates the core functionality of RiffSpace:
- Creating and comparing riffs
- Computing distances in riff space
- Analyzing novelty and evolution
- Visualizing the geometry of rock

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from src.riff import Riff, create_example_riff
from src.space import RiffSpace, RiffCollection
from src.transforms import TransformGroup
from src.metrics import compare_metrics
from src.novelty import NoveltyAnalyzer
from src.visualization import *
from src.pipeline import create_synthetic_dataset

## 1. Creating Riffs

Riffs are represented as sequences of $(\Delta p, \Delta t, a)$ tuples.

In [ ]:
# Create a riff from pitch intervals
riff1 = Riff(
    pitch_intervals=[0, 2, 2, -1, 3, -2],  # Melodic pattern
    durations=[0.5, 0.5, 0.5, 0.5, 1.0, 1.0],  # Rhythmic pattern
    articulations=['palm-mute', 'palm-mute', 'accent', 'normal', 'accent', 'normal'],
    metadata={'name': 'Custom Riff'}
)

print(riff1)
print(f"Interval sequence: {riff1.get_interval_sequence()}")
print(f"Rhythm sequence: {riff1.get_rhythm_sequence()}")

In [ ]:
# Load example riffs
smoke = create_example_riff('smoke_on_the_water')
iron_man = create_example_riff('iron_man')
seven_nation = create_example_riff('seven_nation_army')

print(f"Smoke on the Water: {smoke}")
print(f"Iron Man: {iron_man}")
print(f"Seven Nation Army: {seven_nation}")

## 2. Distance Metrics

Compare riffs using multiple distance metrics.

In [ ]:
# Compare two riffs with all metrics
distances = compare_metrics(smoke, iron_man)

print("Distance between 'Smoke on the Water' and 'Iron Man':")
for metric, dist in distances.items():
    print(f"  {metric:20s}: {dist:.3f}")

## 3. Riff Space & Transformations

Work with equivalence classes under transformations.

In [ ]:
# Create riff space
space = RiffSpace(metric='edit_distance', normalize=True)

# Add riffs
space.add_riffs([smoke, iron_man, seven_nation])

print(space)
print(f"\nDistance matrix:")
print(space.distance_matrix())

In [ ]:
# Apply transformations
transform_group = TransformGroup()
print(f"Available transformations: {transform_group}")

# Generate equivalence class
equivalents = transform_group.apply_all(smoke)
print(f"\nGenerated {len(equivalents)} equivalent riffs")

# Compare with and without transformations
dist_direct = space.distance(smoke, iron_man, use_transforms=False)
dist_equiv = space.distance(smoke, iron_man, use_transforms=True)

print(f"\nDirect distance: {dist_direct:.3f}")
print(f"Equivalence class distance: {dist_equiv:.3f}")

## 4. Temporal Novelty Analysis

**Key Question:** When was rock music most innovative?

In [ ]:
# Create synthetic dataset with temporal information
riffs = create_synthetic_dataset(n_riffs=200, year_range=(1960, 2025))

# Build collection
collection = RiffCollection(space=RiffSpace(metric='dtw'))
for riff in riffs:
    collection.add(riff)

print(collection)

In [ ]:
# Compute novelty scores
analyzer = NoveltyAnalyzer(space=collection.space)
scores = analyzer.analyze_collection(collection)

print(f"Computed novelty for {len(scores)} riffs")
print(f"\nMost novel riffs:")
for score in analyzer.most_novel_riffs(k=5):
    print(f"  Year {score.year}: novelty={score.novelty:.3f}")

In [ ]:
# Plot novelty timeline
fig = plot_novelty_timeline(analyzer, smoothing_window=3)
plt.show()

# Find innovation peaks
peaks = analyzer.find_peak_innovation_periods()
print(f"\nInnovation peaks detected in: {[year for year, _ in peaks]}")

## 5. Genre Geometry

Do genres form distinct manifolds in riff space?

In [ ]:
# Visualize riff space in 2D
try:
    fig = plot_riff_space_2d(collection, method='umap', color_by='genre')
    plt.show()
except ImportError:
    print("UMAP not installed. Install with: pip install umap-learn")

In [ ]:
# Compare genre distributions
fig = plot_genre_comparison(collection)
plt.show()

## 6. Era Comparison

Compare innovation across rock history.

In [ ]:
# Define eras
eras = {
    'Classic Rock (60s-70s)': (1960, 1979),
    'Hair Metal (80s)': (1980, 1989),
    'Grunge/Alt (90s)': (1990, 1999),
    'Modern Rock (2000+)': (2000, 2025)
}

# Compare
era_novelties = analyzer.compare_eras(eras)
print("Average novelty by era:")
for era, novelty in era_novelties.items():
    print(f"  {era:25s}: {novelty:.3f}")

# Plot
fig = plot_era_comparison(analyzer, eras)
plt.show()

## 7. Detecting Discontinuities

Find years with unusually high innovation (genre transitions?).

In [ ]:
# Detect discontinuities
discontinuities = analyzer.detect_discontinuities(threshold_std=1.5)

print("Years with unusually high innovation (>1.5 std above mean):")
for year, z_score in discontinuities[:10]:
    print(f"  {year}: z-score = {z_score:.2f}")

## 8. Influence Networks

Build a graph connecting each riff to its nearest prior riff.

In [ ]:
# Build influence network
edges = analyzer.influence_network(threshold=2.0)
print(f"Found {len(edges)} influence connections")

# Visualize (requires networkx)
try:
    fig = plot_influence_network(analyzer, threshold=2.0)
    plt.show()
except ImportError:
    print("NetworkX not installed. Install with: pip install networkx")

## Next Steps

1. **Load real data:** Use MIDI files from your rock riff collection
2. **Experiment with metrics:** Try different distance functions
3. **Tune transformations:** Customize the equivalence group
4. **Deep learning:** Train a learned metric with Siamese networks
5. **Paper time:** Write up your findings!